[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tml_p1_rl/blob/main/01_stage1_verify.ipynb)

# 1단계. 이해와 검증 — Flow-GRPO on 2D 8-Gaussians

설계: `DESIGN.md`, 코드: `fg8.py` (ARIS `modern_diffusion_post_training_tutorial` §3, §7.1 그대로).

**런타임: CPU 그대로 두세요.** 모델이 2D 입력 MLP이고 배치가 수백 점이라 GPU는 커널 실행·전송 오버헤드가 계산보다 큽니다.
전체 노트북 예상 시간: 수 분 ~ 15분 (Colab CPU 2코어 기준 추정).

In [ ]:
import os, sys, time, dataclasses
if not os.path.exists("fg8.py"):
    !git clone -q https://github.com/HisameOgasahara/tml_p1_rl.git
    %cd tml_p1_rl
import numpy as np, torch, matplotlib.pyplot as plt
import fg8
torch.set_num_threads(os.cpu_count())
print("torch", torch.__version__, "| threads", torch.get_num_threads())
base = fg8.get_base()          # 처음 한 번 RF 사전학습(~1분), 이후 base.pt 재사용

In [ ]:
SEEDS = [0, 1, 2]

def run_all(cfgs):
    out = {}
    for name, c in cfgs.items():
        t0 = time.time()
        out[name] = [fg8.run(base, dataclasses.replace(c, seed=s), name, verbose=(s == 0)) for s in SEEDS]
        print(f"{name}: {time.time() - t0:.0f}s")
    return out

def final(runs, key):
    return np.array([r["ode"][-1][key] for r in runs])

def table(runs_by_name, keys=("reward", "target_mass", "off_support", "within_std")):
    print(f"{'run':<16}" + "".join(f"{k:>16}" for k in keys) + f"{'train s':>10}{'NFE':>12}")
    base_m = next(iter(runs_by_name.values()))[0]["ode"][0]
    print(f"{'base':<16}" + "".join(f"{base_m[k]:>16.4f}" for k in keys))
    for name, runs in runs_by_name.items():
        cells = "".join(f"{final(runs, k).mean():>9.4f}±{final(runs, k).std():<6.4f}" for k in keys)
        print(f"{name:<16}{cells}{np.mean([r['time'][-1] for r in runs]):>10.1f}{runs[0]['nfe'][-1]:>12,}")

def curves(runs_by_name, x="it", key="reward", ax=None, ylim=None, start=0, label=None):
    """key: metric name or a function of the eval dict; non-finite values are dropped; start: first iteration"""
    ax = ax or plt.gca()
    get = key if callable(key) else (lambda m: m[key])
    for name, runs in runs_by_name.items():
        keep = np.array(runs[0]["it"]) >= start
        xs = np.array(runs[0][x])[keep]
        ys = np.array([[get(m) for m in r["ode"]] for r in runs], dtype=float)[:, keep]
        ys[~np.isfinite(ys)] = np.nan
        ax.plot(xs, np.nanmean(ys, 0), label=name)
        ax.fill_between(xs, np.nanmin(ys, 0), np.nanmax(ys, 0), alpha=0.2)
    ax.set_xlabel(x); ax.set_ylabel(f"ODE {label or key}"); ax.legend()
    if ylim: ax.set_ylim(*ylim)

def scatter(ax, pts, title):
    pts = np.asarray(pts); ax.scatter(pts[:, 0], pts[:, 1], s=2, zorder=2)
    m = fg8.MODES.numpy()   # hollow target markers behind the points, so a collapsed cluster stays visible
    ax.scatter(m[list(fg8.TARGET), 0], m[list(fg8.TARGET), 1], s=120, facecolors="none", edgecolors="r", zorder=1)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(title)

## 1-① 작게 구현한 게 논문대로인가 (학습 없음)

- base가 8봉우리를 재현 (봉우리별 질량 ≈ 1/8, 3σ 밖 < 10%)
- θ = old에서 ratio = 1 (학습 코드 경로로 재계산)
- `gauss_logp` = torch `Normal` log-prob 차원 합 (§3.3)
- KL: θ = ref에서 0, 닫힌형 두 가지(평균 차 / v 차) = torch `kl_divergence` (§3.4)
- base의 SDE(T=10) vs ODE(T=40): 기록만 (Q24: 엄밀한 일치 아님)

In [ ]:
chk = fg8.check_implementation(base)
print("PASS 1-①")
print("ratio max |rho-1|:", chk["ratio_max_dev"], "| KL rel err:", chk["kl_rel_err"])
for k in ("base_ode", "base_sde"):
    m = chk[k]; print(f"{k}: masses {np.round(m['masses'], 3)}  off-support {m['off_support']:.3f}  within-std {m['within_std']:.3f}")

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
scatter(ax[0], fg8.sample_data(2000), "data")
z = torch.randn(2000, 2)
scatter(ax[1], fg8.ode_sample(base, z, 40), "base ODE (T=40)")
scatter(ax[2], fg8.rollout_sde(base, z, 10, 0.7)[0][-1], "base SDE (T=10, a=0.7)")
plt.show()

## 1-② 메커니즘이 작동하는지, 비용 순서대로

1. 위 1-① (학습 없음, 초)
2. 업데이트 1회: A>0이면 전이 평균 μ_θ가 실제 샘플된 x_{k+1} 쪽으로, A<0이면 반대로 (§3.6)
3. 전체 학습 (아래 1-③)

In [ ]:
one = fg8.check_one_update(base)
print("PASS 1-②  |mu - x_{k+1}| before -> after:", {k: tuple(round(x, 6) for x in v) for k, v in one.items()})

## 1-③ 재현 판단

baseline = base 모델 (같은 평가 노이즈, ODE 40스텝). 지표(§8): ODE raw reward, 봉우리별 질량, 3σ 밖 비율, 봉우리 내부 std.

| 논문 주장 | toy 판정 |
|---|---|
| §3.7 SDE로 학습해도 ODE 평가 보상이 오른다 | 모든 시드에서 최종 ODE reward > base |
| §3.7 KL이 품질을 지킨다 | β=0 대비 β=0.04가 3σ 밖 비율·봉우리 내부 std 수축이 덜함 |
| §3.5 T_train=10이 40과 비슷한 보상을 더 싸게 | KL 0에서 T10 vs T40: 최종 reward 차이, 학습 시간·NFE 비교 (KL 0.04 쌍은 참고용: 손실이 스텝 평균이라 T40의 실효 KL이 약 1/4) |

In [ ]:
runs = run_all({
    "grpo_kl0.04": fg8.Cfg(),
    "grpo_kl0":    fg8.Cfg(beta_kl=0.0),
    "grpo_T40":    fg8.Cfg(T_train=40),
    "grpo_T40_kl0": fg8.Cfg(T_train=40, beta_kl=0.0),
})

In [ ]:
table(runs)
mass0 = lambda m: m["masses"][0]
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
curves(runs, key="within_std", ax=ax[0]); ax[0].axhline(fg8.SIGMA, ls="--", c="k")
ax[0].set_title("within-std (dashed = data sigma)")
curves(runs, key=mass0, label="mode-0 mass", ax=ax[1]); ax[1].set_title("mode-0 mass (0.5 = both targets kept)")
curves(runs, key="reward", ax=ax[2]); curves(runs, key="off_support", ax=ax[3])
plt.show()
fig, ax = plt.subplots(1, len(runs), figsize=(4.3 * len(runs), 4))
for a_, (name, rs) in zip(ax, runs.items()):
    scatter(a_, rs[0]["final_samples"], f"{name} (seed 0)")
plt.show()


In [ ]:
b = runs["grpo_kl0.04"][0]["ode"][0]
r10, r0, r40, r40_0 = runs["grpo_kl0.04"], runs["grpo_kl0"], runs["grpo_T40"], runs["grpo_T40_kl0"]

c1 = (final(r10, "reward") > b["reward"]).all()
print(f"[§3.7 보상 상승]  base {b['reward']:+.4f} -> {final(r10, 'reward').round(4)}  => {'재현' if c1 else '재현 안 됨'}")

for k in ("off_support", "within_std", "target_mass"):
    print(f"[§3.7 KL 효과] {k:<12} beta=0.04 {final(r10, k).mean():.4f}   beta=0 {final(r0, k).mean():.4f}   base {b[k]:.4f}")
off_lower = final(r10, "off_support").mean() < final(r0, "off_support").mean()
std_closer = abs(final(r10, "within_std").mean() - fg8.SIGMA) < abs(final(r0, "within_std").mean() - fg8.SIGMA)
print(f"  -> off_support: beta=0.04 쪽이 {'낮음' if off_lower else '낮지 않음'}"
      f" | within_std: beta=0.04 쪽이 데이터 σ에 {'더 가까움' if std_closer else '더 멀음'}")

def t35(a, b_, tag):
    print(f"[§3.5 T_train, {tag}] final reward T10 {final(a, 'reward').mean():+.4f} vs T40 {final(b_, 'reward').mean():+.4f}"
          f" | train time {np.mean([r['time'][-1] for r in a]):.0f}s vs {np.mean([r['time'][-1] for r in b_]):.0f}s"
          f" | NFE {a[0]['nfe'][-1]:,} vs {b_[0]['nfe'][-1]:,}")
t35(r0, r40_0, "KL 0 (판정용)")
t35(r10, r40, "KL 0.04 (참고: T40 실효 KL ≈ 1/4)")
